In [0]:
from pyspark.sql.functions import col, concat_ws, regexp_replace, split, upper
import os
import sys
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import importlib
import utils.custom_utils
importlib.reload(utils.custom_utils)
from utils.custom_utils import transformations

In [0]:
current_dir = os.getcwd()
print(current_dir)
sys.path.append(current_dir)

In [0]:
def transform_customers(df):
    df = df.withColumn('domain', split(col('email'), '@')[1])
    df = df.withColumn('phone_number', regexp_replace('phone_number', r'[^0-9]', ''))
    df = df.withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))
    df = df.drop('first_name', 'last_name')
    return df

In [0]:
def transform_vehicles(df):
    df = df.withColumn("make", upper(col("make")))
    return df
    

In [0]:
def transform_trips(df):
    df = df.drop("start_location","end_location","payment_method")
    return df

In [0]:
def transform_passthrough(df):
    return df

In [0]:
def transform_drivers(df):
    df = df.withColumn("phone_number", regexp_replace("phone_number", r'[^0-9]', ""))
    df = df.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    df = df.drop("first_name", "last_name")
    return df

In [0]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

obj = transformations()

def process_table(table, key, custom_transform):
    try:
        logger.info(f"Silver processing started: {table}")

        # Reading bronze
        df = spark.read.table(f"pysparkdbt.bronze.{table}")

        # custom ingestion
        df = custom_transform(df)

        # common
        df = obj.trim_strings(df)
        df = obj.handle_nulls(df, key)
        df = obj.dedup(df, [key], "last_updated_timestamp")
        df = obj.transform_timestamp(df)

        # upsert
        if not spark.catalog.tableExists(f"pysparkdbt.silver.{table}"):
            df.write.format("delta").mode("append").saveAsTable(f"pysparkdbt.silver.{table}")
        else:
            obj.upsert(spark, df, [key], table, "last_updated_timestamp")

        logger.info(f"Silver processing completed: {table}")

    except Exception as e:
        logger.error(f"Silver processing failed for {table}: {e}")
        raise

In [0]:
tables_config = [
    ("customers", "customer_id", transform_customers),
    ("drivers",   "driver_id",   transform_drivers),
    ("locations", "location_id", transform_passthrough),
    ("vehicles",  "vehicle_id",  transform_vehicles),
    ("trips",     "trip_id",     transform_trips),
    ("payments",  "payment_id",  transform_passthrough),
]

for table, key, custom_transform in tables_config:
    process_table(table, key, custom_transform)

In [0]:
tables = ["customers", "drivers", "locations", "vehicles", "trips", "payments"]

In [0]:
for t in tables:
    print(f"Table: {t}")
    count = spark.sql(f"SELECT COUNT(*) FROM pysparkdbt.silver.{t}").collect()[0][0]
    print(f"{t}: {count}")